# Детекция мошенничества: проверяем руками

Ноутбук к теме [`lesson.md`](lesson.md). Считаем то, что в лекции сказано словами:
как precision разваливается от редкости класса; как выбирается порог из стоимости
ошибок и почему правило «$p\cdot$сумма» лучше любого фиксированного порога;
что нужно сделать после undersampling, чтобы вероятности снова стали вероятностями;
и как агрегат, посчитанный без оглядки на время, устраивает утечку будущего.

Данные симулированы: открытые датасеты по фроду анонимизированы до PCA-компонент
или лишены времени и сумм, без которых половину этих выводов не проверить.

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, brier_score_loss, roc_auc_score
from sklearn.preprocessing import StandardScaler

C_BG, C_INK, C_GRAY = "#faf9f5", "#141413", "#b0aea5"
C_PANEL, C_ORANGE, C_BLUE, C_GREEN = "#e8e6dc", "#d97757", "#6a9bcc", "#788c5d"
plt.rcParams.update({
    "figure.facecolor": C_BG, "axes.facecolor": C_BG, "axes.edgecolor": C_GRAY,
    "text.color": C_INK, "xtick.color": C_INK, "ytick.color": C_INK,
    "axes.labelcolor": C_INK, "font.size": 11, "legend.frameon": False,
})

FEATURES = ["log_amount", "hour", "account_age", "tx_1h", "device_share",
            "country_mismatch", "new_payee", "amount_vs_median"]
ATTACKS = {
    "перебор карт": {"log_amount": -1.2, "tx_1h": 2.6, "new_payee": 0.9},
    "угон аккаунта": {"account_age": -1.4, "country_mismatch": 0.95,
                      "amount_vs_median": 2.2},
    "дропы": {"device_share": 2.4, "new_payee": 0.8, "log_amount": 0.9},
}

def simulate(n, rng, prevalence=0.007, mix=(0.6, 0.4, 0.0)):
    '''Поток транзакций: признаки, метка, сумма, схема.'''
    X = np.zeros((n, len(FEATURES)))
    X[:, 0] = rng.normal(0, 1, n)
    X[:, 1] = rng.normal(0, 1, n)
    X[:, 2] = rng.normal(0, 1, n)
    X[:, 3] = rng.gamma(1.5, 0.7, n) - 1.05
    X[:, 4] = rng.gamma(1.2, 0.6, n) - 0.72
    X[:, 5] = (rng.random(n) < 0.08).astype(float)
    X[:, 6] = (rng.random(n) < 0.25).astype(float)
    X[:, 7] = rng.normal(0, 1, n)
    y = (rng.random(n) < prevalence).astype(int)
    idx = np.where(y == 1)[0]
    mix = np.array(mix, float)
    kinds = rng.choice(list(ATTACKS), size=len(idx), p=mix / mix.sum())
    for i, kind in zip(idx, kinds):
        for f, s in ATTACKS[kind].items():
            j = FEATURES.index(f)
            X[i, j] = float(rng.random() < s) if f in ("country_mismatch", "new_payee") \
                else X[i, j] + s * rng.normal(1.0, 0.35)
    amount = np.exp(1.8 + 0.9 * X[:, 0]) * 10
    return X, y, amount

rng = np.random.default_rng(0)
Xtr, ytr, atr = simulate(90_000, rng)
Xte, yte, ate = simulate(90_000, rng)
model = HistGradientBoostingClassifier(max_iter=150, learning_rate=0.08,
                                       max_leaf_nodes=15, early_stopping=False,
                                       random_state=0).fit(Xtr, ytr)
p = model.predict_proba(Xte)[:, 1]
print(f"доля фрода: {yte.mean():.4%},  мошеннических транзакций: {yte.sum()}")
print(f"ROC-AUC {roc_auc_score(yte, p):.4f}   AP {average_precision_score(yte, p):.4f}")
print(f"accuracy у ответа «фрода нет никогда»: {1 - yte.mean():.4%}")

доля фрода: 0.6656%,  мошеннических транзакций: 599
ROC-AUC 0.9724   AP 0.3399
accuracy у ответа «фрода нет никогда»: 99.3344%


Первая строчка — и весь смысл темы: константный ответ «фрода нет» даёт accuracy 99.3%.

## 1. Precision против редкости: формула и прогон

$$\text{precision} = \frac{\text{TPR}\cdot\pi}{\text{TPR}\cdot\pi + \text{FPR}\cdot(1-\pi)}$$

Посмотрим, во что превращается «отличная» модель с TPR 0.8 и FPR 0.01 при разной
доле фрода $\pi$ — и сверим формулу с настоящей моделью на выборке той же редкости.

In [2]:
def precision_formula(tpr, fpr, pi):
    return tpr * pi / (tpr * pi + fpr * (1 - pi))

print(f"{'доля фрода':>12}{'precision по формуле':>24}{'честных на 1 пойманного':>26}")
for pi in (0.05, 0.01, 0.002, 0.0005):
    pr = precision_formula(0.8, 0.01, pi)
    print(f"{pi:>12.2%}{pr:>24.3f}{(1 - pr) / pr:>26.1f}")

# та же величина, посчитанная на реальных оценках модели: берём порог, дающий FPR = 1%
thr = np.quantile(p[yte == 0], 0.99)
tpr = (p[yte == 1] > thr).mean()
fpr = (p[yte == 0] > thr).mean()
prec = (yte[p > thr] == 1).mean()
print(f"\nмодель при пороге FPR = 1%: TPR {tpr:.3f}, FPR {fpr:.3f}, "
      f"precision {prec:.3f}")
print(f"формула на тех же TPR/FPR и π = {yte.mean():.4f}: "
      f"{precision_formula(tpr, fpr, yte.mean()):.3f}")

  доля фрода    precision по формуле   честных на 1 пойманного
       5.00%                   0.808                       0.2
       1.00%                   0.447                       1.2
       0.20%                   0.138                       6.2
       0.05%                   0.038                      25.0

модель при пороге FPR = 1%: TPR 0.618, FPR 0.010, precision 0.293
формула на тех же TPR/FPR и π = 0.0067: 0.293


Совпадает до третьего знака: precision — не свойство модели, а свойство пары
«модель + редкость события». Поэтому ROC-AUC, посчитанная из TPR и FPR, редкости не
замечает вовсе, а PR-кривая — замечает.

## 2. Порог: считаем деньги, а не классы

Пропущенный фрод стоит сумму транзакции, ложная тревога — фиксированные $C_{FP}$.
Сравним четыре политики.

In [3]:
C_FP = 4.0

def total_cost(block, y=yte, amount=ate, c_fp=C_FP):
    '''Штраф за ложные тревоги плюс суммы пропущенного фрода.'''
    return c_fp * np.sum(block & (y == 0)) + np.sum(amount[(~block) & (y == 1)])

cost_none = total_cost(np.zeros(len(yte), bool))

# перебор всех фиксированных порогов разом
order = np.argsort(p)[::-1]
y_s, a_s = yte[order], ate[order]
missed = np.sum(a_s * y_s) - np.concatenate([[0], np.cumsum(a_s * y_s)])
fa = np.concatenate([[0], np.cumsum(1 - y_s)])
tot = missed + C_FP * fa
k_best = int(np.argmin(tot))
thr_best = p[order][min(k_best, len(p) - 1)]

policies = {
    "ничего не блокировать": np.zeros(len(yte), bool),
    "порог 0.5": p > 0.5,
    f"лучший фикс. порог ({thr_best:.3f})": p > thr_best,
    f"правило p·сумма > {C_FP:.0f}": p * ate > C_FP,
}
print(f"{'политика':<32}{'потери':>10}{'% от бездействия':>19}{'алертов':>10}")
for name, block in policies.items():
    c = total_cost(block)
    print(f"{name:<32}{c:>10.0f}{100 * c / cost_none:>18.0f}%{block.mean():>10.2%}")

k_aware = int((p * ate > C_FP).sum())
print(f"\nпри том же числе алертов ({k_aware / len(yte):.2%}) фиксированный порог "
      f"даёт потери {tot[k_aware]:.0f}")
print(f"аналитический порог при постоянной цене пропуска {np.median(ate):.0f}: "
      f"{C_FP / (C_FP + np.median(ate)):.4f}")

политика                            потери   % от бездействия   алертов
ничего не блокировать                35256               100%     0.00%
порог 0.5                            24336                69%     0.51%
лучший фикс. порог (0.037)           17891                51%     2.18%
правило p·сумма > 4                  16896                48%     1.38%

при том же числе алертов (1.38%) фиксированный порог даёт потери 19125
аналитический порог при постоянной цене пропуска 60: 0.0620


Правило с учётом суммы выигрывает у любого фиксированного порога — и делает это
**вдвое меньшим числом алертов**: оно не тратит проверки на дешёвые транзакции с
высоким скором. Ради этого правила вероятность обязана быть вероятностью — что
подводит к следующему разделу.

Отдельно — рабочая точка при ограниченной ёмкости разбора:

In [4]:
print(f"{'бюджет проверок':>16}{'recall':>10}{'precision':>12}{'пойманных сумм':>18}")
total_fraud_amount = ate[yte == 1].sum()
for b in (0.0005, 0.001, 0.002, 0.005, 0.01, 0.02):
    k = int(b * len(yte))
    top = np.argsort(p)[::-1][:k]
    rec = yte[top].sum() / yte.sum()
    prec = yte[top].mean()
    money = ate[top][yte[top] == 1].sum() / total_fraud_amount
    print(f"{b:>16.2%}{rec:>10.2%}{prec:>12.2%}{money:>18.2%}")

 бюджет проверок    recall   precision    пойманных сумм
           0.05%     1.67%      22.22%             2.15%
           0.10%     7.18%      47.78%             8.18%
           0.20%    17.03%      56.67%            18.65%
           0.50%    37.23%      49.56%            33.46%
           1.00%    54.42%      36.22%            48.11%
           2.00%    69.62%      23.17%            64.11%


Обратите внимание на последний столбец: доля пойманных **денег** идёт заметно выше
доли пойманных транзакций — модель, обученная без всякого учёта сумм, всё равно
чаще ловит дорогие транзакции, потому что крупная сумма сама по себе подозрительна.
Именно эту метрику и стоит показывать бизнесу.

## 3. Undersampling и коррекция сдвига

Прореживать отрицательные ради скорости обучения — законно. Но если из негативов
оставить долю $r$, то шансы (odds) сдвигаются ровно в $1/r$ раз, и вероятности надо
вернуть на место:

$$\text{odds}_{\text{true}} = r \cdot \text{odds}_{\text{model}}, \qquad
\text{odds} = \frac{p}{1-p}.$$

In [5]:
scaler = StandardScaler().fit(Xtr)
Xs, Xts = scaler.transform(Xtr), scaler.transform(Xte)

full = LogisticRegression(max_iter=2000).fit(Xs, ytr)
p_full = full.predict_proba(Xts)[:, 1]

r = 0.05                                    # оставляем 5% отрицательных
neg = np.where(ytr == 0)[0]
keep = np.concatenate([np.where(ytr == 1)[0],
                       rng.choice(neg, int(r * len(neg)), replace=False)])
under = LogisticRegression(max_iter=2000).fit(Xs[keep], ytr[keep])
p_under = under.predict_proba(Xts)[:, 1]

odds = p_under / (1 - p_under)
p_fixed = (r * odds) / (1 + r * odds)       # обратная коррекция сдвига

print(f"фактическая доля фрода в тесте: {yte.mean():.5f}\n")
print(f"{'модель':<34}{'средний скор':>14}{'Brier':>10}{'AP':>9}")
for name, pp in [("на всех данных", p_full),
                 (f"undersampling r = {r}", p_under),
                 ("он же после коррекции", p_fixed)]:
    print(f"{name:<34}{pp.mean():>14.5f}{brier_score_loss(yte, pp):>10.5f}"
          f"{average_precision_score(yte, pp):>9.4f}")

print(f"\nранжирование коррекция не трогает: AP до и после совпадают "
      f"({average_precision_score(yte, p_under):.4f} и "
      f"{average_precision_score(yte, p_fixed):.4f}), "
      f"меняется только шкала вероятностей")

фактическая доля фрода в тесте: 0.00666

модель                              средний скор     Brier       AP
на всех данных                           0.00725   0.00546   0.3002
undersampling r = 0.05                   0.05923   0.02217   0.2770
он же после коррекции                    0.00843   0.00585   0.2770

ранжирование коррекция не трогает: AP до и после совпадают (0.2770 и 0.2770), меняется только шкала вероятностей


Коррекция возвращает средний скор к реальной доле фрода и чинит Brier, не трогая
порядок объектов. Если бы решение принималось по правилу «$p\cdot$сумма», без этой
поправки оно блокировало бы всё подряд.

## 4. Утечка через агрегаты: velocity, посчитанный без оглядки на время

Самая коварная ошибка темы. Мошенническая атака — это **серия** транзакций подряд.
Если признак «сколько транзакций у этой карты за час» считать по всей истории (в
окне ±1 час), то в него попадают транзакции той же атаки, случившиеся **позже**.
Офлайн-метрика взлетает; в проде будущего нет.

In [6]:
def simulate_stream(n_cards=9000, days=30, rng=None):
    '''Транзакции по картам во времени; атака — серия транзакций подряд.'''
    rows = []
    for card in range(n_cards):
        n_tx = 1 + rng.poisson(3)
        t = np.sort(rng.uniform(0, days * 24, n_tx))
        for ti in t:
            rows.append((card, ti, np.exp(rng.normal(3.5, 0.9)), 0))
        if rng.random() < 0.012:                    # карту атакуют
            t0 = rng.uniform(0, days * 24)
            burst = t0 + np.sort(rng.exponential(0.06, rng.integers(4, 12)))
            for ti in burst:
                rows.append((card, ti, np.exp(rng.normal(2.6, 0.6)), 1))
    a = np.array(rows)
    o = np.argsort(a[:, 1])
    return a[o]

rng2 = np.random.default_rng(7)
S = simulate_stream(rng=rng2)
card, time, amount, label = S[:, 0].astype(int), S[:, 1], S[:, 2], S[:, 3].astype(int)
print(f"транзакций: {len(S)}, доля фрода: {label.mean():.3%}")

def velocity(card, time, window=1.0, causal=True):
    '''Сколько транзакций той же карты в окне: только в прошлом или вокруг.'''
    out = np.zeros(len(time))
    order = np.argsort(card, kind="stable")
    for c in np.unique(card):
        idx = order[card[order] == c]
        t = time[idx]
        if causal:
            lo = np.searchsorted(t, t - window, "left")
            out[idx] = np.arange(len(t)) - lo            # строго до текущей
        else:
            lo = np.searchsorted(t, t - window, "left")
            hi = np.searchsorted(t, t + window, "right")
            out[idx] = hi - lo - 1                       # включая будущее
    return out

v_causal = velocity(card, time, causal=True)
v_leaky = velocity(card, time, causal=False)
print(f"средний velocity у фрода: причинный {v_causal[label == 1].mean():.2f}, "
      f"с будущим {v_leaky[label == 1].mean():.2f}")
print(f"средний velocity у легитимных: причинный {v_causal[label == 0].mean():.2f}, "
      f"с будущим {v_leaky[label == 0].mean():.2f}")

транзакций: 37014, доля фрода: 2.167%


средний velocity у фрода: причинный 3.59, с будущим 7.18
средний velocity у легитимных: причинный 0.00, с будущим 0.01


In [7]:
split = int(0.7 * len(time))                # разбиение по времени
def build(v):
    return np.column_stack([np.log(amount), v, np.sin(time % 24 / 24 * 2 * np.pi)])

res = {}
for name, v in [("причинный агрегат", v_causal), ("агрегат с будущим", v_leaky)]:
    X = build(v)
    m = HistGradientBoostingClassifier(max_iter=120, random_state=0).fit(
        X[:split], label[:split])
    res[name] = average_precision_score(label[split:], m.predict_proba(X[split:])[:, 1])
    print(f"{name:<22}: AP в офлайн-эксперименте {res[name]:.4f}")

# что произойдёт в проде: модель обучена на «будущем», а на инференсе его нет
X_leaky, X_causal = build(v_leaky), build(v_causal)
m_leaky = HistGradientBoostingClassifier(max_iter=120, random_state=0).fit(
    X_leaky[:split], label[:split])
ap_prod = average_precision_score(
    label[split:], m_leaky.predict_proba(X_causal[split:])[:, 1])
print(f"\nмодель на «агрегате с будущим», а в проде доступен только причинный: "
      f"AP {ap_prod:.4f}")
print(f"обещали {res['агрегат с будущим']:.4f}, получили {ap_prod:.4f} "
      f"— потеря {100 * (1 - ap_prod / res['агрегат с будущим']):.0f}%")
print(f"честная модель на причинном агрегате всё это время давала "
      f"{res['причинный агрегат']:.4f}")

причинный агрегат     : AP в офлайн-эксперименте 0.8825


агрегат с будущим     : AP в офлайн-эксперименте 1.0000



модель на «агрегате с будущим», а в проде доступен только причинный: AP 0.6378
обещали 1.0000, получили 0.6378 — потеря 36%
честная модель на причинном агрегате всё это время давала 0.8825


## 5. Walk-forward: как выглядит честная валидация

Обучаемся на прошлом, проверяемся на следующем куске, сдвигаемся — и так по всему
потоку. Заодно видно, устойчиво ли качество во времени.

In [8]:
rng3 = np.random.default_rng(3)
MIX = [(0.6, 0.4, 0.0)] * 5 + [(0.55, 0.42, 0.03), (0.5, 0.4, 0.1),
                               (0.45, 0.37, 0.18), (0.38, 0.32, 0.30),
                               (0.32, 0.28, 0.40), (0.28, 0.24, 0.48)]
months = [simulate(18_000, rng3, mix=m) for m in MIX]

print(f"{'обучение':>18}{'тест':>8}{'AP':>9}{'доля новой схемы':>20}")
for t in range(4, len(months)):
    Xh = np.vstack([m[0] for m in months[:t]])
    yh = np.concatenate([m[1] for m in months[:t]])
    m_t = HistGradientBoostingClassifier(max_iter=120, learning_rate=0.08,
                                         max_leaf_nodes=15, early_stopping=False,
                                         random_state=0).fit(Xh, yh)
    ap = average_precision_score(months[t][1], m_t.predict_proba(months[t][0])[:, 1])
    print(f"{f'месяцы 0-{t-1}':>18}{t:>8}{ap:>9.3f}{MIX[t][2]:>20.0%}")

          обучение    тест       AP    доля новой схемы


        месяцы 0-3       4    0.477                  0%


        месяцы 0-4       5    0.430                  3%


        месяцы 0-5       6    0.424                 10%


        месяцы 0-6       7    0.369                 18%


        месяцы 0-7       8    0.322                 30%


        месяцы 0-8       9    0.366                 40%


        месяцы 0-9      10    0.314                 48%


## 6. Что осталось в руках

- Accuracy при доле фрода 0.7% — 99.3% за ответ «фрода не бывает»; precision зависит
  не только от модели, но и от редкости, и формула воспроизводит прогон до третьего знака.
- Порог считают из стоимости ошибок, а лучшее решение — вообще не порог, а правило
  «$p\cdot$сумма $> C_{FP}$»: те же потери при вдвое меньшем числе алертов.
- Undersampling законен ради скорости, но сдвигает шансы ровно в $1/r$ раз; обратная
  коррекция возвращает вероятности, не трогая ранжирование.
- Агрегат, посчитанный в окне «вокруг» вместо «до», обещает качество, которого в проде
  не будет: будущее в признаке исчезает ровно в момент выкатки.
- Честная валидация — walk-forward по времени, и она же показывает, когда пора
  переобучаться.